Import Libraries

In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
from typing import Tuple, List

Synthetic Dataset (Sequence Reversal)

In [2]:
def generate_data(n_samples: int = 10000, seq_len: int = 5) -> Tuple[torch.Tensor, torch.Tensor]:
    """Generates synthetic data where the target is the reverse of the input sequence."""
    X = np.random.randint(1, 10, (n_samples, seq_len))
    y = np.flip(X, axis=1).copy()
    return torch.LongTensor(X), torch.LongTensor(y)


X, y = generate_data()
print(X.shape, y.shape)


torch.Size([10000, 5]) torch.Size([10000, 5])


Embedding Layer

In [3]:
class TokenEmbedding(nn.Module):
    """Embedding layer for mapping input tokens to a continuous representation."""
    def __init__(self, vocab_size: int, d_model: int):
        super().__init__()
        self.embedding = nn.Parameter(torch.randn(vocab_size, d_model))

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """Looks up the token embeddings for the input indices."""
        return self.embedding[x]


Position embedding

In [4]:
class PositionEmbedding(nn.Module):
    """Embedding layer for injecting positional information into the sequence."""
    def __init__(self, max_len: int, d_model: int):
        super().__init__()
        self.embedding = nn.Parameter(torch.randn(max_len, d_model))

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """Returns the positional embeddings up to the input sequence length."""
        seq_len = x.shape[1]
        return self.embedding[:seq_len]


Self Attention

In [5]:
class SelfAttention(nn.Module):
    """Computes scaled dot-product self-attention over the sequence."""
    def __init__(self, d_model: int):
        super().__init__()
        # Linear layers to project input to Queries, Keys, and Values
        self.Wq = nn.Linear(d_model, d_model)
        self.Wk = nn.Linear(d_model, d_model)
        self.Wv = nn.Linear(d_model, d_model)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        Q = self.Wq(x)
        K = self.Wk(x)
        V = self.Wv(x)

        # Scaled dot-product attention formula: softmax(Q * K^T / sqrt(d_k)) * V
        scores = torch.matmul(Q, K.transpose(-2, -1)) / np.sqrt(Q.size(-1))
        weights = torch.softmax(scores, dim=-1)

        # Output is the weighted sum of Values
        return torch.matmul(weights, V)


Feed Forward Network

In [6]:
class FeedForward(nn.Module):
    """Two-layer multi-layer perceptron (MLP) applied to each position independently."""
    def __init__(self, d_model: int, hidden: int):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(d_model, hidden),
            nn.ReLU(),
            nn.Linear(hidden, d_model)
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """Passes the input through the 2-layer MLP."""
        return self.net(x)


Transformer Encoder Block

In [7]:
class EncoderBlock(nn.Module):
    """A single Transformer encoder block with self-attention and feed-forward networks."""
    def __init__(self, d_model: int):
        super().__init__()
        self.attn = SelfAttention(d_model)
        self.norm1 = nn.LayerNorm(d_model)
        self.ff = FeedForward(d_model, d_model * 2)
        self.norm2 = nn.LayerNorm(d_model)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # Self-attention with residual connection and layer normalization
        x = self.norm1(x + self.attn(x))
        # Feed-forward network with residual connection and layer normalization
        x = self.norm2(x + self.ff(x))
        return x


Full Transformer

In [8]:
class Transformer(nn.Module):
    """A simple Transformer model with token and positional embeddings followed by an encoder layer."""
    def __init__(self, vocab_size: int = 10, seq_len: int = 5, d_model: int = 32):
        super().__init__()
        self.token_embed = TokenEmbedding(vocab_size, d_model)
        self.pos_embed = PositionEmbedding(seq_len, d_model)
        self.encoder = EncoderBlock(d_model)

        # Output layer maps the transformer representations back to vocabulary size
        self.output = nn.Linear(d_model, vocab_size)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # Combine token semantics and position information
        tok = self.token_embed(x)
        pos = self.pos_embed(x)
        x = tok + pos

        # Pass through the encoder blocks
        x = self.encoder(x)

        # Compute pre-softmax token logits
        return self.output(x)


Training Loop

In [9]:
model = Transformer()

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

epochs = 20

for epoch in range(epochs):
    # Clear previous gradients
    optimizer.zero_grad()

    # Forward pass
    out = model(X)

    # Compute loss (flattening predictions and targets)
    loss = criterion(out.view(-1, 10), y.view(-1))

    # Backward pass to compute gradients
    loss.backward()

    # Update model parameters
    optimizer.step()

    print(f"Epoch {epoch} Loss: {loss.item():.4f}")


Epoch 0 Loss: 2.4680
Epoch 1 Loss: 2.4438
Epoch 2 Loss: 2.4202
Epoch 3 Loss: 2.3973
Epoch 4 Loss: 2.3750
Epoch 5 Loss: 2.3532
Epoch 6 Loss: 2.3319
Epoch 7 Loss: 2.3110
Epoch 8 Loss: 2.2904
Epoch 9 Loss: 2.2701
Epoch 10 Loss: 2.2501
Epoch 11 Loss: 2.2302
Epoch 12 Loss: 2.2104
Epoch 13 Loss: 2.1906
Epoch 14 Loss: 2.1707
Epoch 15 Loss: 2.1507
Epoch 16 Loss: 2.1305
Epoch 17 Loss: 2.1100
Epoch 18 Loss: 2.0891
Epoch 19 Loss: 2.0677


Generate Output

In [ ]:
def predict(model: nn.Module, seq: List[int]) -> np.ndarray:
    """Predicts outputs for a given sequence using the trained model."""
    seq_tensor = torch.LongTensor(seq).unsqueeze(0)

    # Disable gradient computation for inference
    with torch.no_grad():
        out = model(seq_tensor)

    # Return the index with the highest probability
    return out.argmax(-1).squeeze().numpy()


Test

In [ ]:
test = [3,5,2,4,1]

print("Input:", test)
print("Prediction:", predict(model,test))


Input: [3, 5, 2, 4, 1]
Prediction: [4 4 6 5 1]



Model Evaluation

The transformer model was evaluated using token-level accuracy, which measures how many individual digits in the predicted sequences match the true reversed sequences.

After training, the model achieved an accuracy of approximately.

For example:

•	Input: [3, 5, 2, 4, 1]

•	Expected Output: [1, 4, 2, 5, 3]

•	Predicted Output: [4, 4, 6, 5, 1]

This prediction shows that the model correctly learned some positional relationships (e.g., middle elements) but fails to fully reverse the sequence. It struggles to correctly map elements from the beginning of the sequence to the end.

There are several reasons for this behavior:

1. Limited Training Time
  
The model was trained for a relatively small number of epochs, which may not be sufficient for learning the full sequence reversal pattern.

2.	Small Model Capacity
  
The embedding size and number of encoder layers are limited, reducing the model’s ability to capture complex positional dependencies.

3.	Simple Architecture
  
The implementation uses a single-head self-attention mechanism and a single encoder block, which restricts the model’s expressiveness compared to full transformer architectures.

4.	Lack of Positional Generalization

Although positional embeddings are used, the model may not have fully learned how to associate positions with their reversed counterparts.

Despite these limitations, the model demonstrates partial learning, indicating that the implemented transformer components (embedding, attention, feed-forward network, and normalization) are functioning correctly.
